# BTC 5-Minute Market Analysis

Up/Down token mid prices derived from Polymarket orderbook events in
`data/orderbook_price_log.parquet` (built from the
`predict-quant/poly-btc-orderbook` HuggingFace dataset, 5m windows of 2026-04-19).

- One row per `price_change` event per asset
- `mid = (best_bid + best_ask) / 2`
- `outcome` is `Up` or `Down` (resolved via Polymarket CLOB API)


In [ ]:
import pandas as pd
import pyarrow.parquet as pq

PARQUET = 'data/orderbook_price_log.parquet'
N_WINDOWS = None  # number of windows to load; None = all (287)

pf = pq.ParquetFile(PARQUET)
ws_idx = pf.schema_arrow.get_field_index('window_start')
all_windows = sorted({
    pf.metadata.row_group(i).column(ws_idx).statistics.min
    for i in range(pf.num_row_groups)
})
selected = set(all_windows if N_WINDOWS is None else all_windows[:N_WINDOWS])

# Stream row-groups (one per window) and aggregate to 1s-per-outcome to keep memory small.
parts = []
for i in range(pf.num_row_groups):
    ws = pf.metadata.row_group(i).column(ws_idx).statistics.min
    if ws not in selected:
        continue
    chunk = pf.read_row_group(
        i, columns=['timestamp', 'window_start', 'outcome', 'mid']
    ).to_pandas()
    chunk['ts_sec'] = chunk['timestamp'].astype('int64')
    parts.append(
        chunk.groupby(['window_start', 'ts_sec', 'outcome'], as_index=False)['mid'].mean()
    )

df = pd.concat(parts, ignore_index=True)
df['time'] = pd.to_datetime(df['ts_sec'], unit='s')
df['window_label'] = pd.to_datetime(df['window_start'], unit='s').dt.strftime('%Y-%m-%d %H:%M')

print(f"Loaded {len(df):,} second-level rows across {df['window_start'].nunique()} of {len(all_windows)} windows")
df.head()


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

ordered_windows = sorted(df['window_start'].unique())
n = len(ordered_windows)
cols = 6
rows = (n + cols - 1) // cols
subplot_w, subplot_h = 4.0, 2.4  # per-subplot size in inches

fig, axes = plt.subplots(rows, cols,
                         figsize=(subplot_w * cols, subplot_h * rows),
                         squeeze=False)
axes = axes.flatten()

# Pre-pivot once: index=(window_start, ts_sec), columns=outcome -> mid
pv_all = df.pivot_table(index=['window_start', 'ts_sec'],
                        columns='outcome', values='mid', aggfunc='mean')

for i, ws in enumerate(ordered_windows):
    ax1 = axes[i]
    pv = pv_all.loc[ws].sort_index()
    t = pd.to_datetime(pv.index, unit='s')

    if 'Up' in pv:
        ax1.plot(t, pv['Up'], color='green', alpha=0.85, linewidth=1.0, label='Up mid')
    if 'Down' in pv:
        ax1.plot(t, pv['Down'], color='red', alpha=0.85, linewidth=1.0, label='Down mid')
    ax1.set_ylim(-0.05, 1.05)
    ax1.set_ylabel('Mid', fontsize=7)

    ax2 = ax1.twinx()
    if {'Up', 'Down'}.issubset(pv.columns):
        ax2.plot(t, pv['Up'] - pv['Down'], color='blue', alpha=0.55,
                 linewidth=0.8, linestyle='--', label='Up - Down')
        ax2.axhline(0, color='blue', alpha=0.15, linewidth=0.5)
    ax2.set_ylim(-1.05, 1.05)
    ax2.tick_params(axis='y', labelcolor='blue', labelsize=6)
    ax2.set_ylabel('Up - Down', fontsize=7, color='blue')

    label = pd.to_datetime(ws, unit='s').strftime('%m-%d %H:%M')
    ax1.set_title(label, fontsize=8, fontweight='bold')
    ax1.xaxis.set_major_locator(mdates.MinuteLocator(interval=1))
    ax1.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
    ax1.tick_params(axis='x', rotation=0, labelsize=6)
    ax1.tick_params(axis='y', labelsize=6)
    ax1.grid(True, axis='x', linestyle='--', alpha=0.3, linewidth=0.5)

for j in range(n, len(axes)):
    axes[j].set_visible(False)

plt.suptitle(f'BTC 5-Minute Markets: Up/Down Mid Prices - {n} windows',
             fontsize=14, fontweight='bold', y=1.00)
plt.tight_layout()
plt.show()
